# 🛡️ AML Pipeline: Análisis de la Capa Gold y Trazabilidad de Datos

¡Hola! Como Data Engineer a cargo de la arquitectura de datos de nuestro sistema de Anti-Lavado de Dinero (AML), me gustaría presentarles la justificación y las decisiones de diseño tomadas para la construcción de nuestro pipeline, enfocándonos en la escalabilidad y las transformaciones hacia la **Capa Gold**.

## 🏛️ Arquitectura Medallion y el Pipeline AML
Nuestro pipeline de datos sigue la arquitectura **Medallion (Bronze, Silver, Gold)** para garantizar la calidad, escalabilidad y la trazabilidad de los datos:

1. **Capa Bronze (Raw):** Ingestamos los datos transaccionales en bruto, tal cual vienen de los distintos sistemas core. En esta capa mantenemos un historial inmutable ("append-only") para asegurar que siempre podemos reproducir el estado exacto del sistema en cualquier punto del tiempo.
2. **Capa Silver (Cleansed & Conformed):** Aquí aplicamos las transformaciones críticas de calidad (estandarización de fechas y divisas, limpieza de nulos). Utilizamos estrategias de *Change Data Capture (CDC)* para procesar únicamente los deltas (nuevos registros), optimizando enormemente el tiempo de cómputo en nuestro cluster.
3. **Capa Gold (Business & Analytics):** La capa final de valor. Los datos están enriquecidos, consolidados y etiquetados con scores de riesgo, completamente optimizados y listos para ser consumidos por analistas y algoritmos de Machine Learning.

## 💡 Decisiones de Diseño y Optimizaciones del Pipeline

* **Operaciones Idempotentes:** Todas las transformaciones del pipeline están diseñadas para ser idempotentes. Si un *job* falla y se reinicia automáticamente, no generará datos duplicados ni inconsistencias, garantizando la integridad de la capa Gold.
* **Particionamiento Estratégico:** Hemos particionado las tablas Delta por `fecha_transaccion` y `canal`. Esto permite un *Partition Pruning* (poda de particiones) extremadamente eficiente; cuando los modelos de Machine Learning consultan ventanas de tiempo específicas, el motor ignora los datos irrelevantes acelerando las consultas.
* **Trazabilidad Integral (Data Lineage):** Implementamos un seguimiento estricto del linaje de los datos. Cada transacción en la capa Gold posee *metadata* (como `batch_id` y `source_system`) que apunta directamente a su origen en Bronze, cumpliendo con los estrictos requerimientos regulatorios en auditorías AML.
* **Evolución Dinámica de Esquemas (Schema Evolution):** Aprovechamos las capacidades transaccionales de Delta Lake para permitir que el pipeline se adapte automáticamente si agregamos nuevas columnas (por ejemplo, un nuevo vector biométrico) sin interrumpir los procesos productivos existentes.

A continuación, utilizaremos visualizaciones avanzadas con `plotly` para entender cómo nuestro pipeline transforma los datos, extrae insights de riesgo y cómo fluye el capital a través de nuestra arquitectura.

## 📊 1. Materialización de la Capa Gold (Datos de Prueba)
Generaremos un dataset que representa el estado final de las transacciones ya procesadas y enriquecidas por nuestro pipeline, incluyendo las marcas de riesgo identificadas.

In [0]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import random
from datetime import datetime, timedelta

def generate_mock_gold_data(num_records=1500):
    np.random.seed(42)
    random.seed(42)
    
    data = []
    current_date = datetime.now()
    
    # 1. Pitufeo (Structuring)
    for _ in range(40):
        client_id = f"C_PITUFO_{random.randint(100, 999)}"
        for i in range(random.randint(3, 5)):
            data.append({
                'id_transaccion': f"TXN_{len(data)+1}",
                'id_cliente': client_id,
                'fecha': current_date - timedelta(hours=random.randint(1, 72)),
                'monto_usd': random.uniform(1500, 2999), 
                'canal': 'Efectivo',
                'patron_sospechoso': 'Pitufeo (Structuring)',
                'es_fraude': True,
                'riesgo_score': random.uniform(0.7, 0.95)
            })
            
    # 2. Lavador Crypto
    for _ in range(30):
        client_id = f"C_CRYPTO_{random.randint(100, 999)}"
        for i in range(random.randint(5, 10)):
            data.append({
                'id_transaccion': f"TXN_{len(data)+1}",
                'id_cliente': client_id,
                'fecha': current_date - timedelta(hours=i*48), 
                'monto_usd': random.uniform(1000, 2499),
                'canal': 'Exchange_Crypto',
                'patron_sospechoso': 'Salto Crypto (KYT)',
                'es_fraude': True,
                'riesgo_score': random.uniform(0.8, 0.99)
            })
            
    # 3. Flujo Normal Legítimo
    for _ in range(num_records - len(data)):
        client_id = f"C_NORMAL_{random.randint(1000, 9999)}"
        monto = random.uniform(10, 200) if random.random() > 0.05 else random.uniform(1000, 8000)
        data.append({
            'id_transaccion': f"TXN_{len(data)+1}",
            'id_cliente': client_id,
            'fecha': current_date - timedelta(days=random.randint(1, 30)),
            'monto_usd': monto,
            'canal': random.choice(['Transferencia', 'Tarjeta', 'Efectivo']),
            'patron_sospechoso': 'Normal',
            'es_fraude': False,
            'riesgo_score': random.uniform(0.01, 0.2)
        })
        
    df_gold = pd.DataFrame(data)
    df_gold['monto_usd'] = df_gold['monto_usd'].round(2)
    df_gold['fecha'] = pd.to_datetime(df_gold['fecha'])
    return df_gold

# Datos listos y refinados
df_gold = generate_mock_gold_data(1500)
display(df_gold.head(5))

## 🔄 2. Flujo de Capitales (Sankey Diagram)
Este diagrama avanzado de flujo (*Sankey*) ilustra espectacularmente cómo los volúmenes de dinero transitan desde los distintos canales de origen hasta los patrones y clasificaciones que nuestro motor AML identificó en la capa Gold. Es la mejor forma de auditar dónde se concentra el riesgo.

In [0]:
def plot_money_flow_sankey(df):
    # Agrupamos los flujos de dinero por Canal y Patrón
    flujos = df.groupby(['canal', 'patron_sospechoso'])['monto_usd'].sum().reset_index()
    
    # Extraemos nodos únicos
    nodos_origen = list(flujos['canal'].unique())
    nodos_destino = list(flujos['patron_sospechoso'].unique())
    todos_los_nodos = nodos_origen + nodos_destino
    
    # Creamos mapa de índices para el diagrama
    mapa_nodos = {nodo: i for i, nodo in enumerate(todos_los_nodos)}
    
    # Definimos Enlaces (Links)
    source = flujos['canal'].map(mapa_nodos).tolist()
    target = flujos['patron_sospechoso'].map(mapa_nodos).tolist()
    value = flujos['monto_usd'].tolist()
    
    # Asignamos colores estéticos según el riesgo (Rojo=Fraude, Azul=Normal, Amarillo=Crypto)
    colores_enlaces = []
    for patron in flujos['patron_sospechoso']:
        if 'Normal' in patron:
            colores_enlaces.append('rgba(46, 145, 229, 0.4)')
        elif 'Pitufeo' in patron:
            colores_enlaces.append('rgba(255, 75, 75, 0.6)')
        else:
            colores_enlaces.append('rgba(242, 183, 5, 0.6)')
            
    fig = go.Figure(data=[go.Sankey(
        node = dict(
          pad = 20,
          thickness = 25,
          line = dict(color = "black", width = 0.5),
          label = todos_los_nodos,
          color = "#333333"
        ),
        link = dict(
          source = source,
          target = target,
          value = value,
          color = colores_enlaces
        )
    )])
    
    fig.update_layout(
        title_text="Trazabilidad del Flujo de Capitales: Canal de Origen vs Clasificación AML", 
        font_size=13, 
        template='plotly_dark'
    )
    fig.show()

plot_money_flow_sankey(df_gold)

## 🎯 3. Análisis de Composición del Riesgo (Sunburst Chart)
Exploración interactiva y jerárquica del volumen transaccional. Esta gráfica nos responde instantáneamente: **¿Qué porcentaje de nuestro volumen total es fraude, en qué patrón delictivo se concentra y a través de qué canales se está moviendo el dinero ilícito?**

In [0]:
def plot_fraud_sunburst(df):
    # Formateamos los labels para un gráfico más amigable
    df_sunburst = df.copy()
    df_sunburst['clasificacion_global'] = df_sunburst['es_fraude'].map({True: 'Alerta AML (Fraude)', False: 'Flujo Legítimo'})
    
    fig = px.sunburst(
        df_sunburst, 
        path=['clasificacion_global', 'patron_sospechoso', 'canal'], 
        values='monto_usd',
        title='Jerarquía del Riesgo AML: Desglose por Patrón y Canal',
        color='clasificacion_global',
        color_discrete_map={
            'Alerta AML (Fraude)': '#FF4B4B',
            'Flujo Legítimo': '#2E91E5'
        },
        template='plotly_dark'
    )
    
    fig.update_layout(margin = dict(t=50, l=0, r=0, b=0))
    fig.show()

plot_fraud_sunburst(df_gold)

## 📈 4. Distribución Espacial del Score de Riesgo (3D Scatter)
Para entender mejor la sensibilidad de nuestras métricas, podemos visualizar los *Scores de Riesgo* generados en 3 dimensiones: **Tiempo**, **Monto de Transacción** y el **Nivel de Riesgo (0 al 1)**.

In [0]:
def plot_risk_3d(df):
    fig = px.scatter_3d(
        df, 
        x='fecha', 
        y='monto_usd', 
        z='riesgo_score',
        color='patron_sospechoso',
        title='Superficie de Riesgo Transaccional en la Capa Gold',
        size='monto_usd', # El tamaño de la burbuja depende del monto
        opacity=0.8,
        template='plotly_dark',
        color_discrete_map={
            'Normal': '#2E91E5',
            'Pitufeo (Structuring)': '#FF4B4B',
            'Salto Crypto (KYT)': '#F2B705'
        }
    )
    
    fig.update_layout(
        scene=dict(
            xaxis_title='Timeline',
            yaxis_title='Monto (USD)',
            zaxis_title='Score de Riesgo'
        ),
        margin=dict(l=0, r=0, b=0, t=40)
    )
    fig.show()

plot_risk_3d(df_gold)

### 📝 Conclusión del Arquitecto de Datos

Nuestra capa **Gold** no solo sirve como un repositorio limpio, sino como un poderoso motor de *insights* dinámico. Las decisiones fundamentales de diseño como el **Particionamiento Estratégico** y la **Idempotencia** aseguran que estas visualizaciones complejas (Sankey, Sunburst, Modelos 3D) puedan ser calculadas sobre **terabytes de datos** en escasos minutos utilizando los clusters de cómputo distribuido de Databricks.

El estricto control del **linaje de datos (Data Lineage)** se preserva intacto a través de todas las capas, lo que nos permite auditar y rastrear cualquiera de los flujos de capitales mostrados en nuestros diagramas de vuelta hasta su registro original en la capa Bronze.